# This is the search space

In [10]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [11]:
import yaml
import numpy as np
import optuna
import optunahub
from datasets import Dataset
from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score
from src.utils import get_pooled_df
from src.data import get_fold_from_disk, DECODED_LABELS

import matplotlib.pyplot as plt

In [12]:
#Let's load the base yaml for consistency
with open(here('configs/base.yaml'), 'r') as f: #I should probably make a dataclass here to deserialize but this works for now
    base_data = yaml.full_load(f)

print(base_data)

{'max_length': 256, 'cv_k': 5, 'cv_seed': 7, 'eval_strategy': 'epoch', 'save_strategy': 'epoch', 'load_best_model_at_end': True, 'metric_for_best_model': 'macro_f1', 'greater_is_better': True, 'early_stopping_patience': 3, 'fp16': True, 'seed': 42, 'report_to': 'none', 'optim': 'adamw_torch_fused', 'learning_rate': 3.22738e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.06784, 'warmup_ratio': 0.01656, 'num_train_epochs': 15}


In [13]:
full_df = get_pooled_df()

# Single fold: get_fold_from_disk(full_df, fold=0)
# All folds:   get_fold_from_disk(full_df)
train_folds, val_folds = get_fold_from_disk(full_df)

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)
    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data['max_length'],
        )
    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# Handle both single fold and all folds
if isinstance(train_folds, list):
    train_ds = [tokenize_fold(df) for df in train_folds]
    val_ds = [tokenize_fold(df) for df in val_folds]
    for i in range(len(train_ds)):
        print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")
else:
    train_ds = tokenize_fold(train_folds)
    val_ds = tokenize_fold(val_folds)
    print(f"train={len(train_ds)} val={len(val_ds)}")

Map: 100%|██████████| 132/132 [00:00<00:00, 13607.83 examples/s]

Fold 0: train=530 val=133
Fold 1: train=530 val=133
Fold 2: train=530 val=133
Fold 3: train=531 val=132
Fold 4: train=531 val=132


In [14]:
# All folds
train_folds, val_folds = get_fold_from_disk(full_df)

# Single fold
#train_folds, val_folds = get_fold_from_disk(full_df, fold=0)


In [15]:
#TPE visualizer https://hub.optuna.org/visualization/tpe_acquisition_visualizer/
module = optunahub.load_module(
    package="visualization/tpe_acquisition_visualizer",
)
tpe_acquisition_visualizer = module.TPEAcquisitionVisualizer()


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

In [16]:
def objective(trial):
    lr = trial.suggest_float("learning_rate", 2.5e-5, 3.0e-5, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [4,8,16])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.08)
    num_epochs = trial.suggest_int("num_train_epochs", 18,30)

    f1_scores = []
    for fold in range(5):
        model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/all_folds04/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data['eval_strategy'],
            save_strategy=base_data['save_strategy'],
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=base_data['greater_is_better'],
            fp16=base_data['fp16'],
            seed=base_data['seed'],
            logging_steps=10,
            report_to=base_data['report_to'],
            save_total_limit=1,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        f1_scores.append(trainer.evaluate()["eval_macro_f1"])

    return np.mean(f1_scores)

In [ ]:
#study definition
#prereques 
db_path = here("results/optuna/optuna_study.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5
print(storage_url) 


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="roberta-hyperparameter-searchspace_all_folds04",
    storage=storage_url,
)


study.optimize(objective, n_trials=50, callbacks=[tpe_acquisition_visualizer])

#Best trails
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")



[I 2026-02-25 19:48:15,480] A new study created in RDB with name: roberta-hyperparameter-searchspace_all_folds04


sqlite:////home/jubacochran/mids266-exaggeration/results/optuna/optuna_study.db


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1132.67it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch,Training Loss,Validation Loss,Macro F1
1,1.147145,0.949046,0.252336
2,0.988829,0.956170,0.252336
3,0.914287,0.987381,0.273078
4,0.745233,1.095595,0.356669
5,0.603881,1.811442,0.455382
6,0.275835,2.017262,0.445316
7,0.093441,2.556570,0.443861
8,0.215750,2.827419,0.439769


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.23it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

...

Epoch,Training Loss,Validation Loss,Macro F1
1,1.024449,0.948538,0.255452
2,0.847745,0.926828,0.255452
3,0.962608,0.888554,0.255452
4,0.747768,0.839765,0.352149
5,0.685476,0.933468,0.443785
6,0.373701,1.289173,0.409636
7,0.409440,1.414526,0.511556
8,0.314852,2.132240,0.427308
9,0.016140,2.488124,0.404094
10,0.006871,2.575145,0.433867


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.04it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

[I 2026-02-25 20:59:45,164] Trial 13 finished with value: 0.39238186907966205 and parameters: {'learning_rate': 2.7339191227924046e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.05587985830433144, 'warmup_ratio': 0.06946313385070374, 'num_train_epochs': 23}. Best is trial 1 with value: 0.511341322514644.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1628.25it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSI

Epoch,Training Loss,Validation Loss,Macro F1
1,0.974625,0.954729,0.252336
2,0.931107,0.956811,0.252336
3,0.898520,1.048620,0.252336
4,0.781791,1.044507,0.314668
5,0.629034,1.236650,0.380216
6,0.347307,1.515889,0.293865
7,0.298019,1.782619,0.345368
8,0.283172,1.786439,0.354174


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#tpe acquistion 
viz_dir = here("results/optuna/tpe_plots_all_folds04")
viz_dir.mkdir(parents=True, exist_ok=True)

param_names = ["learning_rate", "weight_decay", "warmup_ratio", "num_train_epochs","per_device_train_batch_size"]
for param_name in param_names:
    for trial in study.trials:
        if trial.number < n_startup_trials:
            continue
        try:
            fig = tpe_acquisition_visualizer.plot(study, trial.number, param_name)
            fig.savefig(viz_dir / f"{param_name}_trial{trial.number}.png")
            plt.close(fig)
        except Exception as e:
            print(f"Skipped {param_name} trial {trial.number}: {e}")

print(f"TPE acquisition plots saved to {viz_dir}")

Skipped per_device_train_batch_size trial 5: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 6: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 7: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 8: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 9: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 10: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 11: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 12: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 13: 'CategoricalDistribution' object has no attribute 'low'
Skipped per_device_train_batch_size trial 14: 'CategoricalDistribution' object has no a